# Assembly101 Hand Pose Visualization

This notebook loads one egocentric HMC view, inspects the matching pose JSON, overlays available hand landmarks, and writes a short annotated video. Run the JSON inspection cell before the rendering cells so the view and hand layout can be verified.

In [19]:
from __future__ import annotations

import json
import re
from pathlib import Path

import cv2
import numpy as np

VIDEO_ID = "nusar-2021_action_both_9012-a16_9012_user_id_2021-02-01_162904"
MAX_FRAMES = 2500

# The notebook lives one level below the repository root.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
VIDEO_DIR = REPO_ROOT / "data" / "raw_videos" / "recordings" / VIDEO_ID
POSE_PATH = REPO_ROOT / "data" / "raw_videos" / "poses@60fps" / f"{VIDEO_ID}.json"
OUTPUT_PATH = REPO_ROOT / "output_visualization.mp4"

video_paths = sorted(VIDEO_DIR.glob("HMC_*_mono10bit.mp4"))
if not video_paths:
    print(f"No HMC video found yet in {VIDEO_DIR}")
else:
    print(f"Selected video: {video_paths[0]}")
    print(f"Found {len(video_paths)} HMC view(s): {[path.name for path in video_paths]}")
print(f"Pose file: {POSE_PATH}")

Selected video: c:\Users\lakho\Desktop\NTU\FYP\JointPoseAction\data\raw_videos\recordings\nusar-2021_action_both_9012-a16_9012_user_id_2021-02-01_162904\HMC_84346135_mono10bit.mp4
Found 4 HMC view(s): ['HMC_84346135_mono10bit.mp4', 'HMC_84347414_mono10bit.mp4', 'HMC_84355350_mono10bit.mp4', 'HMC_84358933_mono10bit.mp4']
Pose file: c:\Users\lakho\Desktop\NTU\FYP\JointPoseAction\data\raw_videos\poses@60fps\nusar-2021_action_both_9012-a16_9012_user_id_2021-02-01_162904.json


In [20]:
if not video_paths:
    raise FileNotFoundError(f"No HMC video found in {VIDEO_DIR}")
if not POSE_PATH.exists():
    raise FileNotFoundError(f"Pose JSON not found: {POSE_PATH}")

VIDEO_PATH = video_paths[0]
with POSE_PATH.open("r", encoding="utf-8") as pose_file:
    pose_data = json.load(pose_file)

if not isinstance(pose_data, list):
    raise TypeError(f"Expected a list of frame annotations, got {type(pose_data).__name__}")

capture = cv2.VideoCapture(str(VIDEO_PATH))
if not capture.isOpened():
    raise RuntimeError(f"OpenCV could not open {VIDEO_PATH}")

video_width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
video_fps = capture.get(cv2.CAP_PROP_FPS) or 60.0
print(f"Loaded {len(pose_data)} pose frames; video is {video_width}x{video_height} at {video_fps:.2f} FPS")

Loaded 26133 pose frames; video is 636x480 at 60.00 FPS


In [21]:
# Preliminary schema inspection: run this before choosing a camera/hand mapping.
first_frame = pose_data[0]
print("Top-level JSON type:", type(pose_data).__name__)
print("Number of frames:", len(pose_data))
print("First-frame keys:", list(first_frame.keys()))

landmarks = first_frame.get("landmarks", {})
print("Landmark container type:", type(landmarks).__name__)
if isinstance(landmarks, dict):
    print("Landmark/view keys:", list(landmarks.keys()))
    for key, value in list(landmarks.items())[:8]:
        print(f"  key={key!r}, type={type(value).__name__}, shape={np.asarray(value).shape}")
else:
    print("Landmark sample shape:", np.asarray(landmarks).shape)

print("Selected camera filename:", VIDEO_PATH.name)
print("Camera ID candidates:", re.findall(r"HMC_([^_]+)", VIDEO_PATH.name))

Top-level JSON type: list
Number of frames: 26133
First-frame keys: ['frame_index', 'timestamp', 'landmarks', 'tracking_confidence']
Landmark container type: dict
Landmark/view keys: ['0', '1']
  key='0', type=list, shape=(21, 3)
  key='1', type=list, shape=(21, 3)
Selected camera filename: HMC_84346135_mono10bit.mp4
Camera ID candidates: ['84346135']


In [22]:
# The local pose JSON stores hands directly under landmarks['0'] and landmarks['1'].
def extract_hands(frame: dict, video_path: Path) -> tuple[dict[str, np.ndarray], str | None]:
    _ = video_path  # Kept in the signature so callers can use the same adapter API.
    landmarks = frame.get("landmarks", {})
    if not isinstance(landmarks, dict):
        return {}, "default_cam"

    key_mapping = {
        "0": "left",
        "1": "right",
        "left": "left",
        "right": "right",
    }
    hands = {}
    for key, hand_name in key_mapping.items():
        if key in landmarks:
            points = as_points(landmarks[key])
            if points is not None:
                hands[hand_name] = points

    return hands, "default_cam"


def as_points(value) -> np.ndarray | None:
    array = np.asarray(value, dtype=np.float32)
    if array.ndim != 2 or array.shape[0] < 21 or array.shape[1] < 2:
        return None
    return array[:21, :]


pose_by_frame = {
    int(frame.get("frame_index", index)): frame
    for index, frame in enumerate(pose_data)
    if isinstance(frame, dict)
}

sample_hands, sample_camera_key = extract_hands(pose_data[0], VIDEO_PATH)
print("Hand container: landmarks['0']/landmarks['1']")
print(f"Detected hand groups: {[(name, points.shape) for name, points in sample_hands.items()]}")

Hand container: landmarks['0']/landmarks['1']
Detected hand groups: [('left', (21, 3)), ('right', (21, 3))]


In [ ]:
# Approximate HMC camera calibration for the 1920x1080 video.
# Replace fx/fy and the extrinsics with official calibration when available.
cx = video_width / 2.0
cy = video_height / 2.0
fx = 1200.0
fy = 1200.0

CAMERA_MATRIX = np.array(
    [
        [fx, 0, cx],
        [0, fy, cy],
        [0, 0, 1],
    ],
    dtype=np.float32,
)
DIST_COEFFS = np.zeros((4, 1), dtype=np.float32)
ROTATION_VECTOR = np.zeros((3, 1), dtype=np.float32)
TRANSLATION_VECTOR = np.zeros((3, 1), dtype=np.float32)
USE_XY_FALLBACK_FOR_3D = False
_DEBUG_COORDINATE_PRINTED = False


def project_points(points: np.ndarray) -> np.ndarray:
    global _DEBUG_COORDINATE_PRINTED
    if points.shape[1] == 2:
        return points[:, :2]

    if not _DEBUG_COORDINATE_PRINTED:
        print(f"[DEBUG] Raw 3D coordinate of joint 0: {points[0]}")
        _DEBUG_COORDINATE_PRINTED = True

    projected, _ = cv2.projectPoints(
        points[:, :3].reshape(-1, 1, 3),
        ROTATION_VECTOR,
        TRANSLATION_VECTOR,
        CAMERA_MATRIX,
        DIST_COEFFS,
    )
    return projected.reshape(-1, 2)


HAND_EDGES = (
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
    (5, 9), (9, 13), (13, 17),
)
HAND_COLORS = {"left": (255, 120, 40), "right": (40, 180, 255), "unknown": (80, 220, 80)}


def draw_hand(frame: np.ndarray, points: np.ndarray, color: tuple[int, int, int]) -> None:
    projected = project_points(points)
    visible = []
    for x, y in projected:
        finite = np.isfinite(x) and np.isfinite(y)
        point = (int(round(x)), int(round(y))) if finite else (-1, -1)
        visible.append(finite and 0 <= point[0] < frame.shape[1] and 0 <= point[1] < frame.shape[0])
    for start, end in HAND_EDGES:
        if visible[start] and visible[end]:
            cv2.line(frame, tuple(projected[start].astype(int)), tuple(projected[end].astype(int)), color, 2, cv2.LINE_AA)
    for index, (x, y) in enumerate(projected):
        if visible[index]:
            cv2.circle(frame, (int(round(x)), int(round(y))), 4, color, -1, cv2.LINE_AA)


def draw_hands(frame: np.ndarray, annotation: dict) -> int:
    hands, _ = extract_hands(annotation, VIDEO_PATH)
    for name, points in hands.items():
        draw_hand(frame, points, HAND_COLORS.get(name, HAND_COLORS["unknown"]))
    return len(hands)

In [24]:
START_FRAME = 1000  # Approx. 16 seconds into a 60 FPS recording.
OUTPUT_FRAME_COUNT = 300

if START_FRAME < 0:
    raise ValueError("START_FRAME must be non-negative")
if START_FRAME >= int(capture.get(cv2.CAP_PROP_FRAME_COUNT)):
    raise ValueError(f"START_FRAME {START_FRAME} is beyond the end of the video")

capture.set(cv2.CAP_PROP_POS_FRAMES, START_FRAME)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
writer = cv2.VideoWriter(
    str(OUTPUT_PATH),
    cv2.VideoWriter_fourcc(*"mp4v"),
    video_fps,
    (video_width, video_height),
)
if not writer.isOpened():
    capture.release()
    raise RuntimeError(f"OpenCV could not create {OUTPUT_PATH}")

frames_written = 0
frames_with_hands = 0
try:
    while frames_written < OUTPUT_FRAME_COUNT:
        success, frame = capture.read()
        if not success:
            break

        source_frame_index = START_FRAME + frames_written
        annotation = pose_by_frame.get(source_frame_index)
        if annotation is not None:
            frames_with_hands += int(draw_hands(frame, annotation) > 0)
        else:
            cv2.putText(
                frame,
                f"No pose annotation for source frame {source_frame_index}",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 0, 255),
                2,
                cv2.LINE_AA,
            )

        writer.write(frame)
        frames_written += 1
        if frames_written % 50 == 0:
            print(f"Processed {frames_written}/{OUTPUT_FRAME_COUNT} output frames")
finally:
    capture.release()
    writer.release()

if frames_written == 0:
    raise RuntimeError("No video frames were written")
print(f"Saved {frames_written} frames (source {START_FRAME} onward) to {OUTPUT_PATH}")
print(f"Frames with at least one detected hand group: {frames_with_hands}")
if CAMERA_MATRIX is None and any(
    np.asarray(points).shape[1] >= 3 for points in sample_hands.values()
):
    print("Note: configure camera calibration for a geometrically correct 3D projection.")

[DEBUG] Raw 3D coordinate of joint 0: [ -6.6121063  78.43589   181.4404   ]
Processed 50/300 output frames
Processed 100/300 output frames
Processed 150/300 output frames
Processed 200/300 output frames
Processed 250/300 output frames
Processed 300/300 output frames
Saved 300 frames (source 1000 onward) to c:\Users\lakho\Desktop\NTU\FYP\JointPoseAction\output_visualization.mp4
Frames with at least one detected hand group: 300
Note: configure camera calibration for a geometrically correct 3D projection.
